# 01 · Tensors, Shapes, and How to Actually See Them

Companion to **Chapter 1** of the course. Open `course/ch01-tensors.html` alongside this.

**How to use this notebook:** for every cell marked `PREDICT`, write your answer down
*before* running it. Being wrong and noticing is the entire mechanism by which this
becomes automatic. Skipping the prediction step wastes the exercise.

In [ ]:
import torch
import math

torch.manual_seed(0)

def describe(t, name="tensor"):
    """Everything you should be able to state about a tensor at a glance."""
    print(f"{name}")
    print(f"  shape      {tuple(t.shape)}")
    print(f"  rank       {t.ndim}")
    print(f"  numel      {t.numel():,}")
    print(f"  dtype      {t.dtype}")
    print(f"  bytes      {t.numel() * t.element_size():,}")
    print(f"  stride     {t.stride()}")
    print(f"  contiguous {t.is_contiguous()}")

describe(torch.randn(2, 4, 3), "x = randn(2, 4, 3)")

## 1 · The flat stream underneath

A tensor is one contiguous run of numbers. The shape is an *interpretation* laid over it.
This is the fact that makes `reshape`, `view` and `transpose` make sense.

In [ ]:
x = torch.arange(24).reshape(2, 4, 3)
print(x)
print("\nstrides:", x.stride())
print("\nThe LAST axis varies fastest: 0,1,2 then jump a row.")
print("offset of x[i,j,k] = i*12 + j*3 + k*1  -- those are the strides.")

i, j, k = 1, 2, 1
flat_offset = i * 12 + j * 3 + k * 1
assert x[i, j, k].item() == torch.arange(24)[flat_offset].item()
print(f"\nx[{i},{j},{k}] = {x[i,j,k].item()} = flat[{flat_offset}]  ✓")

## 2 · PREDICT — indexing

**Rule: an integer index removes an axis; a slice keeps it.**

Write down all eight shapes before running the next cell.

In [ ]:
x = torch.randn(8, 128, 512)   # (B, T, C)

exprs = [
    ("x[0]",            x[0]),
    ("x[0:1]",          x[0:1]),
    ("x[:, 2]",         x[:, 2]),
    ("x[:, -1]",        x[:, -1]),
    ("x[:, -1:]",       x[:, -1:]),
    ("x[..., 0]",       x[..., 0]),
    ("x[:, ::2, :]",    x[:, ::2, :]),
    ("x[:, :, None]",   x[:, :, None]),
]
for name, t in exprs:
    print(f"{name:18} -> {tuple(t.shape)}")

print("\nThe pair to burn in:")
print("  x[:, -1]  drops the time axis  -> use when you want a plain (B, C) vector")
print("  x[:, -1:] keeps it as size 1   -> use in a generation loop")

## 3 · PREDICT — reductions

**Rule: the `dim` you name is the axis that disappears.**

In [ ]:
x = torch.randn(2, 4, 3)
for expr in ["x.sum(dim=0)", "x.sum(dim=1)", "x.sum(dim=-1)",
             "x.sum(dim=-1, keepdim=True)", "x.mean(dim=(0, 1))",
             "x.softmax(dim=-1)"]:
    print(f"{expr:30} -> {tuple(eval(expr).shape)}")

print("\nsoftmax keeps the shape but NORMALISES along dim:")
p = x.softmax(dim=-1)
print("  sum along dim=-1:", p.sum(-1).flatten().tolist())

### Why `keepdim=True` exists — LayerNorm in three lines

In [ ]:
x = torch.randn(2, 4, 3)

mu = x.mean(-1, keepdim=True)                      # (2, 4, 1)  <- keepdim!
var = x.var(-1, keepdim=True, unbiased=False)      # (2, 4, 1)
xhat = (x - mu) / torch.sqrt(var + 1e-5)
print("with keepdim:   ", tuple(mu.shape), "->", tuple(xhat.shape), "✓")

mu_bad = x.mean(-1)                                # (2, 4)
try:
    (x - mu_bad)
    print("without keepdim: NO ERROR -- but it broadcast the WRONG way!")
    print("                 shape:", tuple((x - mu_bad).shape))
except RuntimeError as e:
    print("without keepdim: RuntimeError:", str(e)[:80])

## 4 · The big one — reshape vs transpose

Same output shape. **Completely different contents.** This is the bug that silently
breaks hand-written multi-head attention.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)
print("\nx.reshape(4, 3)      -- re-cuts the flat stream 0..11:\n", x.reshape(4, 3))
print("\nx.transpose(0,1).reshape(4,3) -- rearranges FIRST:\n",
      x.transpose(0, 1).reshape(4, 3))

assert not torch.equal(x.reshape(4, 3), x.transpose(0, 1).reshape(4, 3))
print("\nSame shape (4,3). Different numbers. Neither raises an error.")

In [ ]:
# Why .view() sometimes refuses
y = torch.randn(2, 3, 4).transpose(1, 2)
print("after transpose:  shape", tuple(y.shape), " stride", y.stride(),
      " contiguous", y.is_contiguous())
try:
    y.view(2, 12)
except RuntimeError as e:
    print("\ny.view(2,12) ->", str(e).split('.')[0])
print("\nFixes: y.contiguous().view(2,12)   or   y.reshape(2,12)")
print("result shape:", tuple(y.reshape(2, 12).shape))

## 5 · Exercise 1.1 — Shape golf

`x = torch.randn(4, 6, 8)`. **Write down every answer, then run.**

The one to think hardest about is `j` — same shape as `b`, different contents.

In [ ]:
x = torch.randn(4, 6, 8)

answers = {
    "a = x.transpose(0, 2)":              x.transpose(0, 2),
    "b = x.reshape(4, 48)":               x.reshape(4, 48),
    "c = x.mean(dim=1)":                  x.mean(dim=1),
    "d = x[:, ::2, :]":                   x[:, ::2, :],
    "e = x.unsqueeze(2)":                 x.unsqueeze(2),
    "f = x.permute(2, 0, 1)":             x.permute(2, 0, 1),
    "g = x[..., :4]":                     x[..., :4],
    "h = x.sum(dim=(0,2), keepdim=True)": x.sum(dim=(0, 2), keepdim=True),
    "i = x.view(4, 6, 2, 4)":             x.view(4, 6, 2, 4),
    "j = x.transpose(1,2).reshape(4,-1)": x.transpose(1, 2).reshape(4, -1),
}
for k, v in answers.items():
    print(f"{k:38} {tuple(v.shape)}")

print("\nb and j have the SAME shape:", tuple(answers['b = x.reshape(4, 48)'].shape))
same = torch.equal(answers['b = x.reshape(4, 48)'],
                   answers['j = x.transpose(1,2).reshape(4,-1)'])
print("...and the same contents?", same, "  <-- THIS is the trap")

## 6 · Memory arithmetic

The numbers that decide what fits on your GPU.

In [ ]:
def fmt(b):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if b < 1024: return f"{b:7.2f} {unit}"
        b /= 1024
    return f"{b:.2f} PB"

print("One activation tensor, (32, 2048, 4096):")
n = 32 * 2048 * 4096
for dt, sz in [("float32", 4), ("bfloat16", 2), ("float8", 1)]:
    print(f"  {dt:9} {fmt(n * sz)}")

print("\nTraining a 7B model with AdamW (Ch 1.6):")
P = 7e9
for name, byt in [("bf16 weights", 2), ("fp32 master", 4),
                  ("Adam m", 4), ("Adam v", 4), ("gradients", 4)]:
    print(f"  {name:14} {fmt(P * byt)}")
print(f"  {'TOTAL':14} {fmt(P * 18)}   <- before a single activation")
print(f"\n  Serving the same model:  {fmt(P * 2)}")

## 7 · Exercise 1.4 — Llama-3-8B by hand

`n_layer=32, d_model=4096, n_head=32, n_kv_head=8, d_ff=14336, V=128256`, untied.

Compute the answers yourself first.

In [ ]:
V, C, L, H, KV, D, FF = 128256, 4096, 32, 32, 8, 128, 14336

embed = V * C
attn = C*(H*D) + 2*C*(KV*D) + (H*D)*C
mlp = 3 * C * FF
per_layer = attn + mlp + 2*C
total = 2*embed + L*per_layer + C          # untied -> 2 embedding matrices

print(f"embedding matrix   {embed:>15,}  ({embed/total:5.1%} of the model)")
print(f"attention / layer  {attn:>15,}")
print(f"mlp / layer        {mlp:>15,}  ({mlp/per_layer:5.1%} of each layer)")
print(f"per layer          {per_layer:>15,}")
print(f"TOTAL              {total:>15,}  = {total/1e9:.2f} B")
assert 7.9e9 < total < 8.1e9, "should be ~8.03B"

print(f"\nresidual stream at B=4, T=2048: {(4,2048,C)}")
print(f"  in bf16: {fmt(4*2048*C*2)} per layer per saved activation")

kv_per_token = 2 * KV * D * 2 * L          # 2(K,V) * heads * dim * bytes * layers
print(f"\nKV cache: {fmt(kv_per_token)} per token")
print(f"  at 128k context, one sequence: {fmt(kv_per_token * 131072)}")
print("  ...which is MORE than the model's own weights. That is Chapter 13.")

## 8 · The five shapes you will see forever

Run this whenever you feel lost. It is the whole course in one cell.

In [ ]:
B, T, C, H, Dh, V = 2, 16, 128, 4, 32, 1000

tokens = torch.randint(0, V, (B, T))
embeds = torch.randn(B, T, C)
q      = torch.randn(B, H, T, Dh)
attnw  = torch.randn(B, H, T, T).softmax(-1)
logits = torch.randn(B, T, V)

for name, t, note in [
    ("tokens",      tokens, "int64 token ids"),
    ("embeddings",  embeds, "THE RESIDUAL STREAM -- same shape at every layer"),
    ("q / k / v",   q,      "after the reshape dance (Ch 6)"),
    ("attn weights",attnw,  "rows sum to 1; quadratic in T"),
    ("logits",      logits, "one distribution per position"),
]:
    print(f"{name:14} {str(tuple(t.shape)):22} {note}")

assert torch.allclose(attnw.sum(-1), torch.ones(B, H, T), atol=1e-6)
print("\nattention rows sum to 1 ✓")

---
## Self-check

You are ready for Chapter 2 when you can answer these **without running code**:

1. `x` is `(8, 128, 512)`. Shape of `x[:, -1, :]`? Of `x[:, -1:, :]`?
2. `x.sum(dim=1)` on `(2,3,4)` — what shape, and why?
3. Why does `x.transpose(1,2).view(...)` sometimes raise, and what are the two fixes?
4. `(32, 2048, 4096)` in bf16 — how many MB?
5. In `(B, T, C)`, which axes are batch-like and which is the feature axis?

<details><summary>Answers</summary>

1. `(8, 512)` and `(8, 1, 512)` — integer drops the axis, slice keeps it.
2. `(2, 4)` — the named dim is collapsed and removed.
3. Transpose makes strides non-descending, so the tensor is non-contiguous and
   `view` cannot express the new shape as a re-label. Fix with `.contiguous().view()`
   or `.reshape()`.
4. 512 MiB. 32·2048·4096 = 268,435,456 elements × 2 bytes.
5. `B` and `T` are batch-like (ops run independently across them); `C` is the
   feature axis (ops mix along it).

</details>

**Next:** `02_matmul_and_einsum.ipynb`